# Model 2 V3: Fine-tuned DistilBERT — Mean Pooling

**Architecture:** DistilBERT mean pooling (avg all token embeddings) + regression head  
**Change from V2:** Mean pooling replaces [CLS] token pooling  
**Hypothesis:** Mean pooling captures more distributed information than [CLS], which was pretrained for classification  

```
V1/V2: DistilBERT → [CLS] token (768-dim) → head → price
V3:    DistilBERT → mean(all tokens) (768-dim) → head → price
```

## vast.ai Setup (chỉ chạy lần đầu)

```bash
pip install uv
uv sync
```

In [ ]:
from pricer.items import Item
from pricer.distilbert_model_v3 import DistilBERTRunnerV3
from pricer.evaluator import evaluate, plot_training_history

## 1. Load Data

In [ ]:
train, val, test = Item.from_hub("SeanSunny/items_full")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

## 2. Setup Model

DistilBERT with mean pooling head.

In [ ]:
runner = DistilBERTRunnerV3(train, val[:1000])
runner.setup(batch_size=64)

## 3. Train

Max 15 epochs, early stopping patience=3.

In [ ]:
history = runner.train(epochs=15, patience=3, warmup_steps=500)

## 4. Training History

In [ ]:
plot_training_history(history, title="DistilBERT V3 (Mean Pooling, batch=64, 15 epochs)")

## 5. Evaluate on 200 Test Samples

In [ ]:
evaluate(runner.inference, test)

## 6. Save Model Weights

In [ ]:
runner.save("distilbert_model_v3.pth")
print("Saved to distilbert_model_v3.pth")

## 7. Sanity Check

In [ ]:
sample = test[0]
pred = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  ${sample.price:.2f}")
print(f"Predict: ${pred:.2f}")
print(f"Error:   ${abs(pred - sample.price):.2f}")